<a href="https://colab.research.google.com/github/Anubhavxd0/diabetes-ai-prediction/blob/main/diabetes_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diabetes Risk Prediction

**Author:** Anubhav Rai  
**Background:** BSc Medical Laboratory Technology (MLT) + physical sciences

A machine-learning model that estimates diabetes risk from routine clinical lab
values, built on the **Pima Indians Diabetes** dataset (768 patients).

This notebook is written to be *methodologically correct*, not just to print a
high accuracy number. The main things it does carefully:

1. Treats physiologically impossible zeros (e.g. `Glucose = 0`) as **missing data**.
2. Splits data **before** any imputation and keeps preprocessing inside a
   scikit-learn `Pipeline`, so statistics are learned from the training set only
   (**no data leakage**).
3. Compares three models with **stratified cross-validation**.
4. Reports **precision, recall, F1 and ROC-AUC** - not accuracy alone - because
   for a screening tool, *missing a real diabetic (a false negative) is the
   costly error*.

> **Disclaimer:** This is an educational project, not a medical device. It must
> not be used to diagnose or treat any individual.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load the data

The Pima Indians Diabetes dataset has 8 diagnostic measurements and a binary
`Outcome` (1 = diabetes within 5 years, 0 = not).

In [ ]:
url = ("https://raw.githubusercontent.com/jbrownlee/Datasets/"
       "master/pima-indians-diabetes.data.csv")

columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
           'Insulin', 'BMI', 'DiabetesPedigree', 'Age', 'Outcome']

df = pd.read_csv(url, names=columns)
print('Shape:', df.shape)
df.head()

## 3. Exploratory data analysis

### 3.1 Class balance
About 35% of patients are diabetic, so the classes are moderately imbalanced.
That is exactly why plain accuracy can be misleading.

In [ ]:
counts = df['Outcome'].value_counts().sort_index()
print(counts)
print(f"Positive (diabetic) rate: {df['Outcome'].mean():.1%}")

### 3.2 The hidden missing values

Several columns contain `0` where a zero is medically impossible - you cannot
have a glucose, blood pressure, BMI, skin thickness or (realistically) insulin
of exactly 0. These zeros are really **unrecorded measurements**. A naive model
treats them as real values and learns from nonsense.

In [ ]:
zero_as_missing = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print('Count of impossible zeros per column:')
print((df[zero_as_missing] == 0).sum())

In [ ]:
# Convert those impossible zeros to NaN so they are handled as missing data.
# IMPORTANT: we do NOT fill them here. Imputation happens inside the pipeline,
# fit on the training data only, to avoid leaking test information.
df[zero_as_missing] = df[zero_as_missing].replace(0, np.nan)
print('Missing values after marking impossible zeros:')
print(df.isna().sum())

## 4. Train / test split (done first!)

We split *before* imputing or scaling, and use `stratify=y` so the train and
test sets keep the same diabetic ratio. Everything the model learns about the
data distribution must come from the training set only.

In [ ]:
X = df.drop(columns='Outcome')
y = df['Outcome'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print('Train:', X_train.shape, ' Test:', X_test.shape)
print('Train positive rate:', round(y_train.mean(), 3))
print('Test  positive rate:', round(y_test.mean(), 3))

## 5. Build a leak-free pipeline and compare models

Each candidate model is wrapped in a `Pipeline` that imputes missing values
(median) and scales features. During cross-validation, these steps are re-fit
on each training fold, so no information leaks from validation folds.

We compare an interpretable baseline (**Logistic Regression**) against two
ensembles (**Random Forest**, **Gradient Boosting**). `class_weight='balanced'`
helps the models take the minority (diabetic) class seriously.

In [ ]:
def build_pipeline(estimator):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', estimator),
    ])

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=3,
        class_weight='balanced', random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

rows = []
for name, est in models.items():
    scores = cross_validate(build_pipeline(est), X_train, y_train,
                            cv=cv, scoring=scoring)
    rows.append({m: scores[f'test_{m}'].mean() for m in scoring} | {'model': name})

cv_results = pd.DataFrame(rows).set_index('model')[scoring]
cv_results.round(3)

## 6. Select the best model and evaluate honestly

We pick the winner by mean cross-validated **ROC-AUC** (a threshold-independent
measure of ranking quality), then fit it on the full training set and evaluate
on the untouched test set.

In [ ]:
best_name = cv_results['roc_auc'].idxmax()
print('Best model by CV ROC-AUC:', best_name)

best_pipe = build_pipeline(models[best_name])
best_pipe.fit(X_train, y_train)

y_pred = best_pipe.predict(X_test)
y_proba = best_pipe.predict_proba(X_test)[:, 1]

print(f'Accuracy : {accuracy_score(y_test, y_pred):.3f}')
print(f'Precision: {precision_score(y_test, y_pred):.3f}')
print(f'Recall   : {recall_score(y_test, y_pred):.3f}')
print(f'F1       : {f1_score(y_test, y_pred):.3f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba):.3f}')

In [ ]:
print(classification_report(y_test, y_pred,
                            target_names=['No Diabetes', 'Diabetes']))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_estimator(
    best_pipe, X_test, y_test, display_labels=['No Diabetes', 'Diabetes'],
    cmap='Blues', ax=ax[0], colorbar=False)
ax[0].set_title(f'Confusion Matrix - {best_name}')

RocCurveDisplay.from_estimator(best_pipe, X_test, y_test, ax=ax[1])
ax[1].plot([0, 1], [0, 1], '--', color='grey')
ax[1].set_title(f'ROC Curve - {best_name}')
plt.tight_layout()
plt.show()

## 7. Which measurements drive the prediction?

Permutation importance measures how much test ROC-AUC drops when each feature
is randomly shuffled - a model-agnostic, honest importance measure. For a
clinician, this is the interpretable part: it should line up with medical
intuition (glucose, BMI and age tend to dominate).

In [ ]:
perm = permutation_importance(best_pipe, X_test, y_test, n_repeats=20,
                              random_state=RANDOM_STATE, scoring='roc_auc')
order = perm.importances_mean.argsort()

plt.figure(figsize=(7, 4))
plt.barh(X.columns[order], perm.importances_mean[order], color='#2c7fb8')
plt.xlabel('Drop in ROC-AUC when shuffled')
plt.title(f'Feature importance - {best_name}')
plt.tight_layout()
plt.show()

## 8. Save the model

We save the **entire pipeline** (imputer + scaler + model), not just the bare
classifier. That guarantees new data is preprocessed exactly like the training
data at prediction time.

In [ ]:
import joblib

artifact = {'pipeline': best_pipe, 'features': list(X.columns), 'model_name': best_name}
joblib.dump(artifact, 'diabetes_pipeline.joblib')
print('Saved diabetes_pipeline.joblib')

## 9. Predict for a new patient

Unknown measurements are entered as 0 and treated as missing, just like in
training - so the pipeline imputes them instead of trusting a fake zero.

In [ ]:
def predict_risk(patient: dict):
    row = pd.DataFrame([[patient.get(c, np.nan) for c in X.columns]], columns=X.columns)
    row[zero_as_missing] = row[zero_as_missing].replace(0, np.nan)
    prob = best_pipe.predict_proba(row)[0, 1]
    band = 'Low' if prob < 0.30 else ('Moderate' if prob < 0.60 else 'High')
    return prob, band

example = {'Pregnancies': 6, 'Glucose': 148, 'BloodPressure': 72,
           'SkinThickness': 35, 'Insulin': 0, 'BMI': 33.6,
           'DiabetesPedigree': 0.627, 'Age': 50}

prob, band = predict_risk(example)
print(f'Estimated diabetes risk: {prob:.1%}  ->  {band} risk')

## 10. Limitations & honest caveats

- **Population:** the dataset is women of Pima Indian heritage, aged 21+. Results
  do not necessarily transfer to men, children, or other populations.
- **Size:** only 768 records - small by modern standards.
- **Missing data:** roughly half the `Insulin` values were missing and imputed;
  treat insulin-driven predictions with extra caution.
- **Not a diagnosis:** this is a screening/education demo. Real diagnosis needs a
  clinician and confirmatory testing (e.g. fasting glucose, HbA1c).

**Possible next steps:** probability-threshold tuning to prioritise recall,
calibration curves, SHAP explanations, and validation on an Indian patient cohort.